# Teardown

Se presentarán el desglose de las arquitecturas empleadas en este estudio.
Los modelos son: ResNet-18, EfficientNet-B0 y DenseNet-121


## Lo que comparten los tres modelos
Los tres siguen el mismo esquema general de una red convolucional de clasificación:


```
radiografía → [extractor de características: capas convolucionales] → mapa de características
            → [pooling global] → vector → [capa final] → 2 salidas (normal, neumonía)
```

```mermaid
flowchart TB
    IN["Radiografía de tórax<br/>224×224×1 (escala de grises)"]
    RGB["Replicar a 3 canales y normalizar con ImageNet<br/>224×224×3"]

    subgraph STEM["Stem · reduce 4 veces"]
        direction TB
        C1["Conv 7×7, 64 filtros, paso 2<br/>BatchNorm + ReLU<br/>112×112×64"]
        MP["Max-pooling 3×3, paso 2<br/>56×56×64"]
        C1 --> MP
    end

    subgraph L1["layer1 · 56×56×64"]
        direction LR
        B11["BasicBlock"] --> B12["BasicBlock"]
    end

    subgraph L2["layer2 · 28×28×128"]
        direction LR
        B21["BasicBlock ↓2<br/>atajo con conv 1×1"] --> B22["BasicBlock"]
    end

    subgraph L3["layer3 · 14×14×256"]
        direction LR
        B31["BasicBlock ↓2<br/>atajo con conv 1×1"] --> B32["BasicBlock"]
    end

    subgraph L4["layer4 · 7×7×512"]
        direction LR
        B41["BasicBlock ↓2<br/>atajo con conv 1×1"] --> B42["BasicBlock"]
    end

    GAP["Pooling global promedio<br/>7×7×512 → vector de 512"]
    FC["Capa lineal 512 → 2<br/>(reemplaza la original de 1000 clases)"]
    SM["Softmax<br/>P(normal), P(neumonía)"]
    OUT{"P(neumonía) ≥ umbral?"}
    NEU["Neumonía"]
    NOR["Normal"]
    CAM["Grad-CAM usa la salida de layer4<br/>(mapa de 7×7)"]

    IN --> RGB --> STEM --> L1 --> L2 --> L3 --> L4 --> GAP --> FC --> SM --> OUT
    OUT -- "sí" --> NEU
    OUT -- "no" --> NOR
    L4 -.-> CAM

    classDef entrada fill:#eef4fc,stroke:#2a78d6,color:#1a1a19
    classDef salida fill:#fdf0ea,stroke:#eb6834,color:#1a1a19
    classDef nota fill:#f5f5f3,stroke:#b0b0aa,color:#555555,stroke-dasharray:4 3
    class IN,RGB entrada
    class FC,SM,OUT,NEU,NOR salida
    class CAM nota
```

### Detalle del bloque residual (BasicBlock)

Cada etapa tiene 2 bloques. El primer bloque de layer2, layer3 y layer4 reduce la resolución a la mitad y duplica los canales; por eso su atajo necesita una convolución 1×1 para que las dimensiones coincidan antes de sumar.

```mermaid
flowchart LR
    subgraph A["BasicBlock con atajo identidad"]
        direction LR
        XA["x"] --> A1["Conv 3×3<br/>BatchNorm + ReLU"] --> A2["Conv 3×3<br/>BatchNorm"] --> SA(("+")) --> RA["ReLU"] --> YA["y = F(x) + x"]
        XA -- "atajo identidad" --> SA
    end

    subgraph B["BasicBlock con reducción ↓2"]
        direction LR
        XB["x<br/>H×W×C"] --> B1["Conv 3×3, paso 2<br/>BatchNorm + ReLU"] --> B2["Conv 3×3<br/>BatchNorm"] --> SB(("+")) --> RB["ReLU"] --> YB["y<br/>H/2×W/2×2C"]
        XB -- "atajo" --> P["Conv 1×1, paso 2<br/>BatchNorm"] --> SB
    end
```

**Conteo de capas (18):** 1 convolución inicial + 16 convoluciones en los 8 bloques (2 por bloque) + 1 capa lineal. Las convoluciones 1×1 de los atajos no se cuentan.
